# Build a semantic search engine

**AIPI 510, Week 13 activity.** This completes the course starter with a public-domain corpus, overlapping chunks, embeddings, a persistent Chroma index, and a Streamlit app. Run from this repository folder with Python 3.12 after installing requirements.txt.


## 1. Dataset and chunking

The corpus is Project Gutenberg eBook #1661, *The Adventures of Sherlock Holmes*. Each of its twelve stories is split into 180-word passages with a 35-word overlap. Source and license details are in the README.


In [1]:
from semantic_search import load_stories, chunk_stories
stories = load_stories()
chunks = chunk_stories(stories)
print(f'{len(stories)} stories; {len(chunks)} passages')
print(f'First story: {stories[0][0]}')
print(f'First passage: {chunks[0]["text"][:200]}…')


12 stories; 723 passages
First story: A Scandal In Bohemia
First passage: I. To Sherlock Holmes she is always THE woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emo…


## 2. Embed and store in Chroma

Chroma's default ONNX all-MiniLM-L6-v2 model converts each passage to a 384-dimensional vector. The build_index function persists vectors and story metadata in data/chroma and reuses a complete index.


In [2]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
embedding = DefaultEmbeddingFunction()([chunks[0]['text']])[0]
print(f'Embedding dimensions: {len(embedding)}')
from semantic_search import build_index
print(f'Passages in persistent Chroma index: {build_index()}')


Embedding dimensions: 384
Passages in persistent Chroma index: 723


## 3. Search by meaning

The query uses the same embedding model. Chroma ranks passages by cosine distance; displayed similarity is 1 minus distance.


In [3]:
from semantic_search import search
results = search('What happened to the blue carbuncle?', top_k=3)
for result in results:
    print(f'{result["story"]} | passage {result["chunk_number"]} | similarity {result["similarity"]:.3f}')
    print(result['text'][:220] + '…\n')


The Adventure Of The Blue Carbuncle | passage 17 | similarity 0.364
a plumber, was accused of having abstracted it from the lady's jewel-case. The evidence against him was so strong that the case has been referred to the Assizes. I have some account of the matter here, I believe." He rum…

The Adventure Of The Blue Carbuncle | passage 22 | similarity 0.356
family is now devouring." When the commissionaire had gone, Holmes took up the stone and held it against the light. "It's a bonny thing," said he. "Just see how it glints and sparkles. Of course it is a nucleus and focus…

The Adventure Of The Blue Carbuncle | passage 16 | similarity 0.349
the Countess of Morcar's blue carbuncle!" I ejaculated. "Precisely so. I ought to know its size and shape, seeing that I have read the advertisement about it in The Times every day lately. It is absolutely unique, and it…



## 4. Run the app

Run streamlit run app.py for a search interface with story filtering and adjustable result count. The README explains deployment to Streamlit Community Cloud.
